# 05 Tracker実行・群れ評価

04_simulation.ipynbで保存した全Dataset・全カメラ配置へTrackerを適用し，群れ中心，非観測時間，予測成功率を比較する．TrackerにはObservationだけを渡し，Ground Truthは追跡完了後にEvaluatorだけが読む．

## 1．一括実験を設定する

DATASET_IDSとLAYOUT_IDSをNoneにすると，Case内で利用可能な全条件を実行する．一部だけ試す場合はIDのリストを指定する．

In [ ]:
CASE_ID = "MiyotaMtBase_gsi_3km_v3"  # 使用する農場Case
BENCHMARK_ID = "miyota-group-tracking-v1"  # 条件別結果をまとめるID
DATASET_IDS = None  # Noneなら全Dataset，または["dataset-id"]
LAYOUT_IDS = None  # Noneなら全配置，または["layout-id"]
tracker_methods = ["last_observation", "constant_velocity"]  # 全て実行するTracker
TRACKER_SEED = 42  # Trackerの再現用seed
OVERWRITE = False  # 既存RunとBenchmarkを暗黙に上書きしない
ENABLE_RERUN = False  # Trueなら代表RunをRerun記録へ出力する


## 2．Trackerと群れ判定を設定する

群れへの加入距離と離脱距離を分けることで，境界付近で群れ判定が反転することを防ぐ．真の群れIDや未来位置は設定しない．

In [ ]:
tracker_settings = {
    "confirmation_hits": 3,  # 新規Trackを確定する観測回数
    "confirmation_window": 5,  # 確定判定に使う最大step数
    "association_gate_m": 8.0,  # 個体Trackの対応候補距離m
    "ambiguity_margin_m": 1.5,  # 対応候補差が小さい時の曖昧判定m
    "lost_after_misses": 1,  # LOSTへ移る未観測回数
    "retire_after_s": 15.0,  # RETIREDへ移る未観測秒数
    "exit_confirmation_s": 5.0,  # 牧場退出候補を確定する秒数
    "group_distance_m": 20.0,  # 同じ群れへ加入できる距離m
    "group_leave_distance_m": 25.0,  # 群れ離脱を検討する距離m
    "group_confirmation_s": 3.0,  # 群れ加入を確定する継続秒数
    "group_leave_confirmation_s": 3.0,  # 群れ離脱を確定する継続秒数
}

## 3．実行条件を確認する

In [ ]:
from pathlib import Path
import os
import pandas as pd
from field_environment import find_repository_root
from field_benchmark import discover_benchmark_conditions

REPOSITORY_ROOT = find_repository_root()
CASES_ROOT = Path(os.environ.get("WILDLIFE_CASES_ROOT", REPOSITORY_ROOT / "cases"))
available_conditions = discover_benchmark_conditions(CASES_ROOT, CASE_ID)
selected_conditions = tuple(
    (dataset_id, layout_id)
    for dataset_id, layout_id in available_conditions
    if (DATASET_IDS is None or dataset_id in DATASET_IDS)
    and (LAYOUT_IDS is None or layout_id in LAYOUT_IDS)
)
if not selected_conditions:
    raise ValueError("指定条件に一致するDatasetとカメラ配置がない．")
pd.DataFrame(selected_conditions, columns=["Dataset", "Camera layout"])


## 4．全条件でTrackerとEvaluatorを実行する

既存Runがある場合は再利用する．新しい結果を作る場合は新しいBENCHMARK_IDを使用し，過去結果を保持する．

In [ ]:
from field_benchmark import run_benchmark_matrix

batch_result = run_benchmark_matrix(
    CASES_ROOT,
    case_id=CASE_ID,
    benchmark_id=BENCHMARK_ID,
    tracker_methods=tracker_methods,
    tracker_settings=tracker_settings,
    tracker_seed=TRACKER_SEED,
    conditions=selected_conditions,
    overwrite=OVERWRITE,
)
results = batch_result.condition_results
print(f"保存先: {batch_result.benchmark_directory}")
print(f"実行Run数: {len(results):,}")

## 5．群れ予測結果を比較する

主結果は10 m以内の予測成功率である．5 m・20 mは閾値に対する感度確認として併記する．

In [ ]:
display_columns = [
    "dataset_id", "layout_id", "tracker_method", "visibility_coverage_rate",
    "hidden_state_count", "mean_hidden_position_error_m",
    "prediction_success_rate_5m", "prediction_success_rate_10m",
    "prediction_success_rate_20m", "reacquisition_success_rate",
]
results[display_columns].sort_values(
    ["dataset_id", "layout_id", "tracker_method"]
).reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(10, 7))
for (layout_id, tracker_method), group in results.groupby(["layout_id", "tracker_method"]):
    axis.scatter(
        group["visibility_coverage_rate"],
        group["prediction_success_rate_10m"],
        s=90,
        label=f"{layout_id} / {tracker_method}",
    )
axis.set_xlabel("2D visibility coverage rate")
axis.set_ylabel("Prediction success rate within 10 m")
axis.set_xlim(0.0, 1.0)
axis.set_ylim(0.0, 1.0)
axis.grid(True, alpha=0.3)
axis.legend(fontsize=8)
plt.show()

## 6．非観測時間別の結果を確認する

In [ ]:
duration_tables = []
case_directory = CASES_ROOT / CASE_ID
for row in results.itertuples(index=False):
    path = case_directory / "runs" / row.run_id / "group_duration_summary.parquet"
    table = pd.read_parquet(path)
    if table.empty:
        continue
    table.insert(0, "tracker_method", row.tracker_method)
    table.insert(0, "layout_id", row.layout_id)
    table.insert(0, "dataset_id", row.dataset_id)
    duration_tables.append(table)
duration_results = pd.concat(duration_tables, ignore_index=True) if duration_tables else pd.DataFrame()
duration_results

## 7．Rerun 2D記録について

一括評価結果から確認対象のrun_idを選び，保存済みRunをRerunへ読み込む．初期の数値評価では無効のままでよい．

In [ ]:
if ENABLE_RERUN:
    print("Rerun確認対象のrun_idをresults表から選択すること．")
else:
    print("Rerun出力は無効である．数値評価と保存は完了している．")

## 8．評価式と新しいTrackerの追加

- 評価式と妥当性：packages/field-benchmark/README.md
- Tracker追加方法：workspace/tracking/README.md
- 新しいTrackerはworkspace/tracking/<方式名>.pyへcreate_tracker()を実装し，先頭のtracker_methodsへ方式名を追加する．